# 이지희 | Binary XGBoost + 완화형 클래스 가중치

DACON 암종 분류 개인 모델 제출용 재현 코드입니다. 원래 실험 노트북의 **최종 사용 경로**를 별도 파일로 정리한 버전입니다.

- 데이터: `train.csv` (6,201행), `test.csv` (2,546행), `sample_submission.csv` (2,546행)
- 검증: **Binary 프로필 그룹 기준 StratifiedGroupKFold 5-Fold**
- 기록된 내부 검증 결과: **OOF Macro F1 = 0.3453** (기존 실험에서 측정; 아래 셀 실행 시 재계산)
- 모델: `XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, eval_metric="mlogloss", tree_method="hist")`
- 클래스 가중치: 각 Fold의 학습 레이블만으로 `sqrt(N / (K × 클래스 빈도))` 계산

**실행 안내**: Colab에서 `train.csv`, `test.csv`, `sample_submission.csv` 세 파일을 `/` 또는 `/content`에 업로드한 뒤 위에서부터 순서대로 실행하세요. 원본 데이터와 제출 CSV는 GitHub 저장소에 포함하지 않습니다. Test 데이터는 모델 훈련·검증·선택에 사용하지 않고, 학습 데이터에서 정한 열 순서와 고정 Binary 인코딩 규칙만 적용합니다. (XGBoost 버전·실행 환경이 바뀌면 수치가 달라질 수 있습니다.)

## 1. 라이브러리 및 데이터 불러오기

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score

def find_input(filename):
    for folder in (Path('/'), Path('/content'), Path.cwd()):
        path = folder / filename
        if path.is_file():
            return path
    raise FileNotFoundError(f"{filename} 파일을 Colab에 업로드해 주세요.")

train = pd.read_csv(find_input('train.csv'))
test = pd.read_csv(find_input('test.csv'))
sample_submission = pd.read_csv(find_input('sample_submission.csv'))
print('Train:', train.shape, 'Test:', test.shape, '제출 양식:', sample_submission.shape)

## 2. 정답 라벨과 Binary 피처 준비

In [ ]:
# 고정 변환: WT 또는 결측값 -> 0, 그 외 문자열 -> 1
le_subclass = LabelEncoder()
y_subclass = pd.Series(le_subclass.fit_transform(train['SUBCLASS']), index=train.index)
X = train.drop(columns=['SUBCLASS', 'ID'])
X_binary_all = X.fillna('WT').ne('WT').astype('uint8')

# 동일한 Binary 프로필은 항상 같은 Fold 안에 있도록 그룹화
groups_binary = pd.util.hash_pandas_object(X_binary_all, index=False)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
folds_binary = list(cv.split(X, y_subclass, groups_binary))

assert X_binary_all.columns.is_unique
assert y_subclass.nunique() == 26
for fold_no, (tr_idx, va_idx) in enumerate(folds_binary, start=1):
    assert set(groups_binary.iloc[tr_idx]).isdisjoint(set(groups_binary.iloc[va_idx]))
    assert y_subclass.iloc[tr_idx].nunique() == 26
    assert y_subclass.iloc[va_idx].nunique() == 26
    print(f'Fold {fold_no}: train={len(tr_idx)}, valid={len(va_idx)}, 중복 그룹=0')

## 3. 5-Fold 검증 — 완화형 클래스 가중치

In [ ]:
oof_weighted = np.full(len(y_subclass), -1, dtype=int)
weighted_scores = []
num_classes = len(le_subclass.classes_)

for i, (tr_idx, va_idx) in enumerate(folds_binary, start=1):
    Xtr_i = X_binary_all.iloc[tr_idx]
    Xva_i = X_binary_all.iloc[va_idx]
    ytr_i = y_subclass.iloc[tr_idx]
    yva_i = y_subclass.iloc[va_idx]

    counts_i = np.bincount(ytr_i.to_numpy(), minlength=num_classes)
    assert np.all(counts_i > 0)
    weights_i = np.sqrt(len(tr_idx) / (num_classes * counts_i))
    sample_weights_i = weights_i[ytr_i.to_numpy()]

    model_w_i = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        eval_metric='mlogloss',
        tree_method='hist'
    )
    model_w_i.fit(Xtr_i, ytr_i, sample_weight=sample_weights_i)
    preds_i = model_w_i.predict(Xva_i)
    oof_weighted[va_idx] = preds_i
    score_i = f1_score(yva_i, preds_i, average='macro')
    weighted_scores.append(score_i)
    print(f'Fold {i} Macro F1: {score_i:.4f}', flush=True)

assert np.all(oof_weighted >= 0)
weighted_oof_f1 = f1_score(y_subclass, oof_weighted, average='macro')
print(f'5-Fold OOF Macro F1: {weighted_oof_f1:.4f}')

## 4. 전체 Train으로 최종 모델 학습

In [ ]:

import numpy as np
import xgboost as xgb

# 1. 전체 Train 데이터와 정답
X_final = X_binary_all
y_final = y_subclass

# 학습 전 데이터 확인
assert len(X_final) == len(y_final) == 6201
assert X_final.shape[1] == 4384
assert y_final.nunique() == 26
assert X_final.index.equals(y_final.index)

# 2. 전체 Train의 암종별 빈도 계산
final_counts = np.bincount(
    y_final.to_numpy(),
    minlength=26
)

# 3. 완화형 클래스 가중치 계산
final_class_weights = np.sqrt(
    len(y_final) / (26 * final_counts)
)

final_sample_weights = final_class_weights[
    y_final.to_numpy()
]

# 4. 최종 제출용 모델 생성
final_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss",
    tree_method="hist"
)

# 5. 전체 Train 데이터로 학습
final_model.fit(
    X_final,
    y_final,
    sample_weight=final_sample_weights
)

print("=== 최종 제출용 모델 학습 완료 ===")
print("학습 데이터:", X_final.shape)
print("학습된 암종 수:", final_model.n_classes_)


## 5. Test 추론 — 열 순서 정렬 및 같은 Binary 인코딩

In [ ]:

# 1. Test 데이터에서 ID 제거
X_test_raw = test.drop(columns=["ID"])

# 2. Train과 Test의 유전자 열이 같은지 확인
assert X_test_raw.columns.is_unique
assert set(X_test_raw.columns) == set(X_final.columns)

# 3. Train과 동일한 열 순서로 정렬
X_test_raw = X_test_raw.loc[:, X_final.columns]

# 4. 학습 때와 동일한 Binary 인코딩
X_test_binary = (
    X_test_raw.fillna("WT").ne("WT").astype("uint8")
)

# 5. 최종 모델로 암종 예측
test_pred_numeric = final_model.predict(X_test_binary)

# 6. 숫자 라벨을 암종 이름으로 복원
test_pred_labels = le_subclass.inverse_transform(
    test_pred_numeric.astype(int)
)

# 7. 예측 결과 확인
assert len(test_pred_labels) == len(test) == 2546

print("=== Test 예측 완료 ===")
print("예측 데이터:", len(test_pred_labels))
print("유전자 피처:", X_test_binary.shape[1])
print("예측 결과 예시:", test_pred_labels[:5])


## 6. 제출 파일 저장 및 형식 검증

In [ ]:

import pandas as pd

# 원본 제출 양식을 복사
submission_final = sample_submission.copy()

# 모델 예측 결과 입력
submission_final["SUBCLASS"] = test_pred_labels

# 제출 형식 검증
assert submission_final.shape == (2546, 2)
assert submission_final.columns.tolist() == ["ID", "SUBCLASS"]
assert submission_final["ID"].equals(test["ID"])
assert submission_final["SUBCLASS"].notna().all()
assert submission_final["SUBCLASS"].isin(
    le_subclass.classes_
).all()

# CSV 저장
output_path = "/content/jihee_binary_weighted_submission.csv"

submission_final.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

# 저장한 파일을 다시 읽어서 검증
saved_submission = pd.read_csv(output_path)

assert saved_submission.equals(submission_final)

print("=== 제출 파일 생성 완료 ===")
print("저장 위치:", output_path)
print("파일 크기:", saved_submission.shape)
print("\n예측 암종 분포:")
print(saved_submission["SUBCLASS"].value_counts().to_string())


### 참고
- 노트북 실행 시 출력되는 OOF F1은 **학습 데이터 내부 검증 점수**이며, 실제 대회 리더보드 점수가 아닙니다.
- 모델·가중치·인코딩은 기존 제출 CSV를 만든 실험과 같은 설정입니다. **이 정리본 자체의 전체 실행 결과는 별도로 확인해야 합니다.**
- 기존 실험 노트북에는 베이스라인/경량화/오분류 진단 등의 탐색 기록이 보존되어 있습니다. 이 파일은 팀 공유용 실행 경로만 간추린 것입니다.